# Preprocess NBA Data

In [1]:
import os
import pandas as pd
import numpy as np

# Redefine directories
DATA_DIRECTORY = "nba_training_data"
SCORES_DIRECTORY = os.path.join(DATA_DIRECTORY, "scores")

# New directory for processed data
PROCESSED_DIRECTORY = os.path.join(DATA_DIRECTORY, "processed")
os.makedirs(PROCESSED_DIRECTORY, exist_ok=True)

In [2]:
# Define the filepath
data_path = os.path.join(SCORES_DIRECTORY, "raw_nba_player_boxscores_6yr.csv")
if not os.path.exists(data_path):
        print(f"File not found: {data_path}. Please run your fetch script first.")

In [3]:
# Clean the data
def clean_data(raw_data):
        df = raw_data.copy()
        df['GAME_DATE'] = pd.to_datetime(df['GAME_DATE'])
        # Fill missing stat values with 0
        stat_cols = ['PTS', 'REB', 'AST', 'STL', 'BLK', 'TOV', 'MIN', 'FG3M',]
        for col in stat_cols:
                if col in df.columns:
                        # Handle MIN which might be strings like "34:12"
                        if col == 'MIN' and df[col].dtype == 'object':
                                df[col] = df[col].fillna('0:00')
                                df[col] = df[col].apply(lambda x: float(str(x).split(':')[0]) + (float(str(x).split(':')[1]) / 60) if ':' in str(x) else float(x))
                        else:
                                df[col] = df[col].fillna(0).astype(float)
    
        # Extract opponent and home/away flag from MATCHUP
        df['HOME_GAME'] = df['MATCHUP'].str.contains(' vs. ').astype(int)
        df['OPPONENT'] = df['MATCHUP'].str.split(' ').str[-1]
        return df


In [4]:
# Convert conventional stats into fantasy points
def convert_fantasy_points(clean_df):
        df = clean_df.copy()
        df['FANTASY_PTS'] = (
                df['PTS'] * 1.0 +
                df['REB'] * 1.0 +
                df['AST'] * 2 +
                df['STL'] * 4.0 +
                df['BLK'] * 4.0 +
                df['FG3M'] * 1.0 +
                df['FGM'] * 2.0 -
                df['FGA'] * 1.0 +
                df['FTM'] * 1.0 -
                df['FTA'] * 1.0 -
                df['TOV'] * 2.0
        )
        return df

In [5]:
# Create rolling averages(averages using n-past games) for each NBA player
def create_rolling_avgs(fantasy_points_df):
        # Sort data chronologically
        df = fantasy_points_df.sort_values(by=['PLAYER_ID', 'GAME_DATE']).reset_index(drop=True)
        grouped_player = df.groupby('PLAYER_ID')
    
        # 3-Game 
        df['FP_ROLLING_3'] = grouped_player['FANTASY_PTS'].transform(lambda x: x.shift(1).rolling(3, min_periods=1).mean()).fillna(0)
        df['MIN_ROLLING_3'] = grouped_player['MIN'].transform(lambda x: x.shift(1).rolling(3, min_periods=1).mean()).fillna(0)

        # 4-Game
        df['FP_ROLLING_4'] = grouped_player['FANTASY_PTS'].transform(lambda x: x.shift(1).rolling(4, min_periods=1).mean()).fillna(0)
        df['MIN_ROLLING_4'] = grouped_player['MIN'].transform(lambda x: x.shift(1).rolling(4, min_periods=1).mean()).fillna(0)
    
        # 10-Game
        df['FP_ROLLING_10'] = grouped_player['FANTASY_PTS'].transform(lambda x: x.shift(1).rolling(10, min_periods=1).mean()).fillna(0)
        df['MIN_ROLLING_10'] = grouped_player['MIN'].transform(lambda x: x.shift(1).rolling(10, min_periods=1).mean()).fillna(0)

        df['FP_PER_MIN_10'] = np.where(
                df['MIN_ROLLING_10'] > 0, 
                df['FP_ROLLING_10'] / df['MIN_ROLLING_10'], 0
        )
        
        # Non-fantasy Stats Rolling Averages (4-Game)
        normal_stats = ['PTS', 'REB', 'AST', 'STL', 'BLK', 'TOV', 'FG3M']
        for stat in normal_stats:
                df[f'{stat}_ROLLING_4'] = grouped_player[stat].transform(
                lambda x: x.shift(1).rolling(4, min_periods=1).mean()).fillna(0)
        return df

In [6]:
# Add empty statistics for teams / opportunities for statistics to be filled
def add_vacated_stats(rolling_avgs_df):
        df = rolling_avgs_df.copy()
        # Assign expected stats based on recent performance
        df['EXPECTED_MIN'] = df['MIN_ROLLING_4']
        df['EXPECTED_PTS'] = df['PTS_ROLLING_4']
        df['EXPECTED_REB'] = df['REB_ROLLING_4']
        df['EXPECTED_AST'] = df['AST_ROLLING_4']

        # Sum up the active roster's expected output for tonight's game
        team_game_expected = df.groupby(['TEAM_ID', 'GAME_DATE'], observed=False)[
        ['EXPECTED_MIN', 'EXPECTED_PTS', 'EXPECTED_REB', 'EXPECTED_AST']].sum().reset_index()

        team_game_expected.rename(columns={
                'EXPECTED_MIN': 'TEAM_ACTIVE_MIN',
                'EXPECTED_PTS': 'TEAM_ACTIVE_PTS',
                'EXPECTED_REB': 'TEAM_ACTIVE_REB',
                'EXPECTED_AST': 'TEAM_ACTIVE_AST'
        }, inplace=True)

        # Merge expected team outputs back to individual player rows
        df = pd.merge(df, team_game_expected, on=['TEAM_ID', 'GAME_DATE'], how='left')
    
        # Calculate Vacated Stats using standard NBA team per-game averages
        df['TEAM_VACATED_MINUTES'] = (240.0 - df['TEAM_ACTIVE_MIN']).clip(lower=0)
        df['TEAM_VACATED_PTS'] = (115.0 - df['TEAM_ACTIVE_PTS']).clip(lower=0)
        df['TEAM_VACATED_REB'] = (45.0 - df['TEAM_ACTIVE_REB']).clip(lower=0)
        df['TEAM_VACATED_AST'] = (25.0 - df['TEAM_ACTIVE_AST']).clip(lower=0)
    
        # Clean up calculation columns
        df = df.drop(columns=[
                'EXPECTED_MIN', 'EXPECTED_PTS', 'EXPECTED_REB', 'EXPECTED_AST',
                'TEAM_ACTIVE_MIN', 'TEAM_ACTIVE_PTS', 'TEAM_ACTIVE_REB', 
                'TEAM_ACTIVE_AST'])
        return df

In [7]:
# Calculate and add opponent defensive rating(points allowed) and 
# offensive pace(amount of possessions)
def add_opponent_metrics(df):
        df = df.copy()
    
        # 1. Aggregate individual player stats to their Team for every game
        team_game_stats = df.groupby(['GAME_DATE', 'TEAM_ID', 'OPPONENT'], observed=False)[
                ['PTS', 'FGA', 'FTA', 'TOV']].sum().reset_index()

        # 2. Approximate Possessions using FGA, FTA, and TOV
        team_game_stats['POSSESSIONS'] = team_game_stats['FGA'] + (0.44 * team_game_stats['FTA']) + team_game_stats['TOV']

        # 3. Define defense vs. offensive relative to a team
        defense_df = team_game_stats[['GAME_DATE', 'OPPONENT', 'PTS', 'POSSESSIONS']].copy()
        defense_df.rename(columns={
                'OPPONENT': 'TEAM_ID',
                'PTS': 'PTS_ALLOWED',
                'POSSESSIONS': 'PACE_ALLOWED'
        }, inplace=True)

        # 4. Calculate Rolling Averages for the Defense
        # Sort chronologically 
        defense_df = defense_df.sort_values(by=['TEAM_ID', 'GAME_DATE']).reset_index(drop=True)
        grouped_def = defense_df.groupby('TEAM_ID')

        # Opponent Defensive Generosity (Points allowed per game)
        defense_df['OPP_DEF_RTG_10'] = grouped_def['PTS_ALLOWED'].transform(
                lambda x: x.shift(1).rolling(10, min_periods=1).mean()
        ).fillna(115.0) # Fallback to rough league average

        # Opponent Pace (Possessions allowed per game)
        defense_df['OPP_PACE_10'] = grouped_def['PACE_ALLOWED'].transform(
                lambda x: x.shift(1).rolling(10, min_periods=1).mean()
        ).fillna(100.0)

        # 5. Merge the defensive context back into the main player dataframe
        # We join on GAME_DATE and OPPONENT to attach the correct defensive 
        # rating to the player's matchup
        df = df.merge(
                defense_df[['GAME_DATE', 'TEAM_ID', 'OPP_DEF_RTG_10', 'OPP_PACE_10']],
                left_on=['GAME_DATE', 'OPPONENT'],
                right_on=['GAME_DATE', 'TEAM_ID'],
                how='left',
                suffixes=('', '_DEF')
        )

        # Clean up the duplicate TEAM_ID column from the merge
        df = df.drop(columns=['TEAM_ID_DEF'])

        return df

In [8]:
# Identify and drop non-streamers from the dataset (i.e. top 156 players who
# will be normally rostered in a standard 12-person fantasy league). 
# Non-streamers are identified by their top-156 past-season FP, and new players 
# will be removed from the dataset if their average FP from the past 10 games
# crosses the top-120 player cutoff (to prevent star rookies from being
# misidentified as streamers) 
def label_streamers(df, top_n=156, breakout_top_n=120, min_games=10):
        df = df.copy()

        # Define NBA Seasons
        df['SEASON'] = np.where(
                df['GAME_DATE'].dt.month >= 8,
                df['GAME_DATE'].dt.year,
                df['GAME_DATE'].dt.year - 1
        )

        # 1. Define prior-season ranking and in season-performance
        season_player_avg = (
                df.groupby(['SEASON', 'PLAYER_ID'])['FANTASY_PTS']
                .mean()
                .rename('SEASON_AVG_FP')
                .reset_index()
                .sort_values(['PLAYER_ID', 'SEASON'])
        )
    
        # Shift forward: season Y only sees season Y-1 stats
        season_player_avg['PRIOR_SEASON_AVG_FP'] = (
                season_player_avg.groupby('PLAYER_ID')['SEASON_AVG_FP'].shift(1)
        )
        season_player_avg['PRIOR_SEASON_RANK'] = (
                season_player_avg.groupby('SEASON')['PRIOR_SEASON_AVG_FP']
                .rank(ascending=False, method='first')
        )
    
        # Define top-120 player cutoff
        top_breakout_players = season_player_avg[season_player_avg['PRIOR_SEASON_RANK'] <= breakout_top_n]
        season_cutoffs = top_breakout_players.groupby('SEASON')['PRIOR_SEASON_AVG_FP'].min().rename('STAR_CUTOFF')

        # Define in-season performance
        df = df.sort_values(by=['PLAYER_ID', 'GAME_DATE']).reset_index(drop=True)
        grouped = df.groupby(['PLAYER_ID', 'SEASON'])

        # Trailing window shifted by 1 to prevent look-ahead bias
        df['CURRENT_SEASON_TRAILING_FP'] = grouped['FANTASY_PTS'].transform(
                lambda x: x.shift(1).rolling(window=min_games, min_periods=min_games).mean()
        )

        # 2. Merge rankings and cutoffs back to main dataframe
        df = df.merge(
                season_player_avg[['SEASON', 'PLAYER_ID', 'PRIOR_SEASON_RANK']],
                on=['SEASON', 'PLAYER_ID'],
                how='left'
        ).merge(
                season_cutoffs,
                on='SEASON',
                how='left'
        )

        # 3. Determine streamers vs. non-streamers (top-156/120)
        # Non-streamer status uses the 156-player cutoff
        is_drafted_top_tier = df['PRIOR_SEASON_RANK'].fillna(np.inf) <= top_n
    
        # Consistent breakout status uses the 120-player cutoff
        star_bar = df['STAR_CUTOFF'].fillna(35.0)
        is_in_season_breakout = df['CURRENT_SEASON_TRAILING_FP'] >= star_bar

        df['IS_ROSTERED'] = is_drafted_top_tier | is_in_season_breakout

        # Clean up calculation columns
        return df.drop(columns=['CURRENT_SEASON_TRAILING_FP', 'STAR_CUTOFF'], errors='ignore')

In [9]:
# Filters out rostered players (non-streamers/top 156) and consistent in-season
# breakout players (top 120), leaving only valid streamers in the dataset
def filter_out_top_players(df, top_n=156, breakout_top_n=120, min_games=10):
        labeled_df = label_streamers(df, top_n=top_n, breakout_top_n=breakout_top_n, min_games=min_games)
    
         # Filter out rostered players and drop tracking columns cleanly
        streamer_df = labeled_df[~labeled_df['IS_ROSTERED']].drop(
                columns=['SEASON', 'PRIOR_SEASON_RANK', 'IS_ROSTERED'],
                errors='ignore'
        )

        # Ensure contiguous ordering and remove slates with fewer than 2 streamers
        streamer_df = streamer_df.sort_values(by=['GAME_DATE', 'TEAM_ID', 'PLAYER_ID']).reset_index(drop=True)
        slate_sizes = streamer_df.groupby('GAME_DATE')['PLAYER_ID'].transform('count')
        streamer_df = streamer_df[slate_sizes >= 2].reset_index(drop=True)

        print(f"Dropped Top {top_n} baseline & Top {breakout_top_n} breakout players. Rows remaining: {len(streamer_df)} out of {len(df)}")
        return streamer_df


In [10]:
# Parition and separate the datasets into training (2020-2023), 
# validation (2024-25), and testing (2025-26), allowing the model to be tested
# on the more recent seasons with more recent NBA trends
def partition_datasets(streamer_df):
        val_start_date = '2024-09-01'
        test_start_date = '2025-09-01'

        train_df = streamer_df[streamer_df['GAME_DATE'] < val_start_date]
        val_df = streamer_df[(streamer_df['GAME_DATE'] >= val_start_date) & (streamer_df['GAME_DATE'] < test_start_date)]
        test_df = streamer_df[streamer_df['GAME_DATE'] >= test_start_date]

        print("\nPartition Summary:")
        print(f"Training Set:   {len(train_df)} rows")
        print(f"Validation Set: {len(val_df)} rows")
        print(f"Testing Set:    {len(test_df)} rows")
        return train_df, val_df, test_df

In [11]:
# Save the processed data into the correct directories
def save_processed_data(train_df, val_df, test_df):
        train_out = os.path.join(PROCESSED_DIRECTORY, "train_streamers.csv")
        val_out = os.path.join(PROCESSED_DIRECTORY, "val_streamers.csv")
        test_out = os.path.join(PROCESSED_DIRECTORY, "test_streamers.csv")
    
        train_df.to_csv(train_out, index=False)
        val_df.to_csv(val_out, index=False)
        test_df.to_csv(test_out, index=False)
    
        print(f"\nSuccessfully saved processed datasets to {PROCESSED_DIRECTORY}/")


In [12]:
# Setup dataset
main_df = pd.read_csv(data_path)

In [13]:
# Preprocess dataset
print("Cleaning dataset")
main_df = clean_data(main_df)

print("Adding fantasy points")
main_df = convert_fantasy_points(main_df)

print("Calculating rolling averages")
main_df = create_rolling_avgs(main_df)

print("Calculating open minutes")
main_df = add_vacated_stats(main_df)

print("Adding opponent metrics")
main_df = add_opponent_metrics(main_df)

print("Filtering out non-streamers")
streamer_df = filter_out_top_players(main_df)

print("Paritioning the datasets")
train_df, val_df, test_df = partition_datasets(streamer_df)

save_processed_data(train_df, val_df, test_df)
print("Finished preprocessing data")

Cleaning dataset
Adding fantasy points
Calculating rolling averages
Calculating open minutes
Adding opponent metrics
Filtering out non-streamers
Dropped Top 156 baseline & Top 120 breakout players. Rows remaining: 96657 out of 154346
Paritioning the datasets

Partition Summary:
Training Set:   64967 rows
Validation Set: 15951 rows
Testing Set:    15739 rows

Successfully saved processed datasets to nba_training_data/processed/
Finished preprocessing data
